# Richmond HAC — Modified C-Section Moment Connection

**Independent limit-state check — AISC 360-22 (LRFD), ASCE 7-22 demands**

An **HSS 7x4x3/8** beam (centered on the column WP) nests inside an 11.5"-long
**HSS 8x6x3/8** stub that is coped (one web removed) into an open C. The stub is
CJP-welded to a **PL 3/4 x 16 x 6** cover plate, which is fillet-welded (5/16",
all around) to an **HSS 6x6x3/8** column. The beam is shimmed snug (bottom + sides)
with 1/2" end clearance and fastened with **three 1" dia F2280 bolts** (STD holes,
bearing-type, double shear) through the flange sandwich.

**Axes** — x along the beam, y vertical (8"/7" dim), z horizontal (6"/4" dim).

| material | Fy | Fu |
|---|---|---|
| HSS A500 Gr.C | 50 | 62 |
| PL A572-50 | 50 | 65 |
| weld E70 | — | FEXX = 70 |

> **Critical modeling premise — Path A.** The major moment `Muz` is carried as a
> horizontal flange couple through the **bolts in double shear**, *not* by vertical
> bearing in the pocket. This is what keeps the seat flange a `Vuy`-only check
> (Section 9). If any moment rides on vertical bearing, the seat force jumps from
> ~9 k to 30-85 k and the seat-flange conclusion no longer holds.

> Scope: a peer check to compare against the design calcs; not an EOR sign-off.


In [1]:
%%capture

# --- Environment setup -------------------------------------------------
!pip install handcalcs forallpeople numpy pandas

import sys, os
sys.path.append(os.path.abspath('..'))     # project root -> makes `helpers` importable

# One import brings: units (kip, inch, ksi, ...), math (sin, cos, pi, ...),
# sig(), export_notebook(), setup_formatting(), and the handcalcs %%render magic.
from helpers.formatting import *
setup_formatting(3)                          # 3 significant figures + kip-inch unit rendering

## 1  Geometry & fastener data

In [2]:
%%render params
t      = 0.349*inch      # design wall = 0.93 x 3/8 (all HSS)
d_beam = 7*inch
b_beam = 4*inch
d_stub = 8*inch
b_stub = 6*inch
B_col  = 6*inch
L_stub = 11.5*inch
d_b    = 1.0*inch        # bolt diameter
A_b    = 0.785*inch**2   # bolt area
d_h    = 1.0625*inch     # STD hole
n_b    = 3               # bolts
s_p    = 3*inch          # pitch and edge distance
w_leg  = 0.3125*inch     # fillet weld leg
t_pl   = 0.75*inch       # cover plate thickness
b_w    = 6*inch          # weld group width  (z)
d_w    = 16*inch         # weld group depth  (y)


<IPython.core.display.Latex object>

## 2  Materials & factored demands (at the column-CL work point)

In [3]:
%%render params
F_y   = 50*ksi
F_u   = 62*ksi
F_yp  = 50*ksi
F_up  = 65*ksi
F_EXX = 70*ksi
phi   = 0.75             # bolt / weld rupture
M_uz  = 424.8*kip*inch   # 35.4 kip-ft  major moment (about z)
M_uy  = 47.6*kip*inch    # 3.97 kip-ft  minor moment (about y)
T_u   = 8.52*kip*inch    # 0.71 kip-ft  torsion (about x)
P_c   = 11.4*kip         # axial compression
P_t   = 5.8*kip          # axial tension
V_uy  = 8.9*kip          # vertical shear
V_uz  = 1.0*kip          # horizontal shear
F_nv  = 68*ksi           # F2280 (Group 150), threads included (N)
F_nt  = 113*ksi


<IPython.core.display.Latex object>

## 3  Bolts — shear (double shear)

`Muz` resolves to a horizontal flange couple `F_flange = Muz/(d_beam - t)` shared
by the 3 bolts; axial, `Muy` (elastic group), `Vuz` and torsion add in-plane shear.
Worst shear plane, combined as vector components.

In [ ]:
%%render
d_bf     = d_beam - t                    # beam flange centroid separation
F_flange = M_uz / d_bf                   # horizontal flange couple force
c_b      = 3*inch                        # end bolt to group centroid
J_b      = 18*inch**2                    # sum of x squared for the 3 bolts
V_x      = F_flange/n_b + P_c/(n_b*2)    # x shear per plane, couple plus axial
V_z      = M_uy*c_b/J_b/2 + V_uz/(n_b*2) + T_u/d_bf/n_b   # z shear per plane
V_bolt   = (V_x**2 + V_z**2)**0.5        # resultant per shear plane
R_n      = phi*F_nv*A_b                  # capacity per shear plane
DCR_bolt_shear = V_bolt / R_n


## 3b  Bolts — bearing / tearout (per ply, flange t = 0.349)

Governing clear distance is the 3" pitch (`lc = 3 - d_h`). Same for beam and stub
flanges; the 3/4" cover plate is not on the bolt line.

In [ ]:
%%render
l_c    = s_p - d_h                       # clear distance, pitch controls
R_brg  = phi*2.4*d_b*t*F_u               # bearing
R_to   = phi*1.2*l_c*t*F_u               # tearout
R_bt   = min(R_brg, R_to)
DCR_bolt_bearing = V_bolt / R_bt


## 4  Beam tension-flange rupture (AISC F13)

Vertical bolts hole both flanges. With `Fy/Fu = 0.81 > 0.8`, `Yt = 1.1`; since
`Fu*Afn < Yt*Fy*Afg`, holes govern flexure and `Mn = (Fu*Afn/Afg)*Sx`.
`Sx` here is the gross-box value (corner radii ignored) — confirm against the
AISC table; non-governing either way.

In [ ]:
%%render
A_fg  = b_beam*t                         # gross flange (top wall)
A_fn  = (b_beam - d_h)*t                 # net flange at a hole
Y_t   = 1.1
b_i   = b_beam - 2*t
h_i   = d_beam - 2*t
I_x   = (b_beam*d_beam**3 - b_i*h_i**3)/12
S_x   = I_x/(d_beam/2)
M_n   = (F_u*A_fn/A_fg)*S_x              # F13-1 cap (Fu*Afn < Yt*Fy*Afg)
phiM_n = 0.9*M_n
DCR_beam_flange = M_uz/phiM_n


## 5  Fillet weld group — plate to column (5/16", E70)

Elastic weld-line method, perimeter 16 x 6. Critical point is the corner (normal
stress from `Muz` + `Muy` + `P`; in-plane shear is small).

In [ ]:
%%render
L_w    = 2*(b_w + d_w)                    # total weld length
I_xw   = d_w**3/6 + b_w*d_w**2/2          # weld line, about z
S_xw   = I_xw/(d_w/2)
I_yw   = b_w**3/6 + d_w*b_w**2/2          # about y
S_yw   = I_yw/(b_w/2)
f_M    = M_uz/S_xw                        # normal, from Muz
f_My   = M_uy/S_yw                        # normal, from Muy
f_P    = P_c/L_w                          # normal, from axial
f_weld = f_M + f_My + f_P                 # corner resultant, normal governs
t_thr  = 0.707*w_leg
R_nw   = phi*0.6*F_EXX*t_thr              # base capacity per length
DCR_weld_base = f_weld/R_nw


**Chapter K5 effective width.** The transverse (top/bottom) welds on the 6x6
face are only partly effective; `be = (10/(B/t))(Fy*t/(Fyp*tp))*bw`. Reducing the
top/bottom weld to `be` raises the effective stress.

In [ ]:
%%render
b_e     = (10/(B_col/t))*(F_y*t/(F_yp*t_pl))*b_w
I_xwe   = d_w**3/6 + b_e*d_w**2/2
S_xwe   = I_xwe/(d_w/2)
f_Me    = M_uz/S_xwe
f_welde = f_Me + f_My + f_P
DCR_weld_K5 = f_welde/R_nw


## 6  Cover plate — flexural yielding (3/4 x 16, about z)

In [ ]:
%%render
S_p   = t_pl*d_w**2/6
sig_p = M_uz/S_p
DCR_plate = sig_p/(0.9*F_yp)


Plate thickness 3/4" exceeds the ~5/8" needed for the interface yield-line — OK.

## 7  Column HSS wall — beta = bp/B = 6/6 = 1.0

Full-width plate loads the side walls directly (efficient); punching shear need
not be checked at beta = 1. Side-wall yielding under the compression flange force
`C = Muz/d_couple` (approx; assumed bearing length 5", confirm with a clean K check).

In [ ]:
%%render
d_cp   = d_w - 2*inch                     # couple arm, forces near plate edges
C_fl   = M_uz/d_cp                        # compression flange force
N_eff  = 5*inch                           # assumed comp bearing length
R_sw   = 2*F_y*t*N_eff                    # two side walls
DCR_col_wall = C_fl/R_sw


## 8  Stub C-section (8x6x3/8, one 8-web removed)

Flexure about z (2 flanges + 1 web), shear on the remaining web, and St-Venant
torsion of the open section (warping additional — verify combined with flexural
shear given the CJP-restrained end).

In [ ]:
%%render
y_f    = (d_stub - t)/2
h_w    = d_stub - 2*t
I_zs   = 2*(b_stub*t*y_f**2) + t*h_w**3/12
S_zs   = I_zs/(d_stub/2)
phiM_s = 0.9*F_y*S_zs
DCR_stub_flex = M_uz/phiM_s

A_ws   = h_w*t                            # remaining web
phiV_s = 0.6*F_y*A_ws
DCR_stub_shear = V_uy/phiV_s

J_o    = (b_stub + b_stub + h_w)*t**3/3   # open-section torsion constant
tau_t  = T_u*t/J_o                        # St-Venant shear stress


## 9  Stub seat flange — two-way yield line

Beam centered, tight bottom + side shims: the rigid HSS "plug" ties the stub top
and bottom flanges, so a downward seat reaction hinges **both** flanges at the web
line (capacity `2*phi*mp`). Because the flange is CJP'd to the cover plate across
its full width, a pure fold about the web is not admissible — the collapse
mechanism carries a **diagonal yield line** off the fixed CJP corner
(region A rotates about the CJP edge, region B about the web; `internal/(mp*theta)
= a + c + 2h^2/c`, minimized over the diagonal `c`).

First the hand-checkable 1-way + sharing bound: 

In [12]:
%%render
m_p     = F_y*t**2/4                       # plate plastic moment, kip-in per in
phi_b   = 0.9
cap_web = 2*phi_b*m_p                      # both flanges hinge at the web line
z_w     = (b_stub - t)/2                   # remaining web line from centerline
arm     = z_w                              # centered beam, load centroid at z=0
L_brg   = 11*inch                          # shimmed bearing length (x)
m_dem   = V_uy*arm/L_brg                   # web-line demand per length, 1-way
DCR_seat_1way_shared = m_dem/cap_web


<IPython.core.display.Latex object>

Now the full two-way collapse (numeric minimization over the corner diagonal):

In [13]:
import numpy as np
Fy=50.0; tt=0.349; mp=Fy*tt**2/4; phib=0.9
Vuy=8.9
h_tip=(6-tt)/2 + 3.0                       # web CL to open outer edge
a=11.5                                     # flange length (CJP-fixed at xi=0)
eta_lo, eta_hi = (6-tt)/2-2, (6-tt)/2+2    # beam z=-2..+2 mapped from web
xi_lo, xi_hi   = 0.5, 11.5                 # beam-stub overlap
q = Vuy/((xi_hi-xi_lo)*(eta_hi-eta_lo))

N=400
xis=np.linspace(xi_lo,xi_hi,N); etas=np.linspace(eta_lo,eta_hi,N)
XI,ETA=np.meshgrid(xis,etas); dA=(xis[1]-xis[0])*(etas[1]-etas[0])
def qc_1flange(c):                         # single-flange collapse pressure
    inA = XI < c*ETA/h_tip                 # region A rotates about CJP edge
    w   = np.where(inA,(h_tip/c)*XI,ETA)   # w/theta field
    return mp*(a + c + 2*h_tip**2/c)/(np.sum(w)*dA)
cs=np.linspace(1.0,25,600)
qc=min(qc_1flange(c) for c in cs)          # yield-line minimum
DCR_seat_2way = q/(2*phib*qc)              # x2 for flange sharing, phi=0.9

print(f"collapse pressure (1 flange) qc = {sig(qc)} ksi   demand q = {sig(q)} ksi")
print(f"DCR (2-way + two-flange sharing) = {sig(DCR_seat_2way)}")


collapse pressure (1 flange) qc = 0.396 ksi   demand q = 0.202 ksi
DCR (2-way + two-flange sharing) = 0.284


**Result — DCR ~ 0.28.** The seat clears comfortably and is insensitive to
bearing length (stays ~0.25-0.28 down to a 5" footprint). The two-way action plus
flange sharing removes the earlier 1-way concern. Guardrails: the ~2x sharing
assumes the top clamp / bolt pretension is engaged (F2280 installs to pretension);
and this remains a `Vuy`-only check under Path A.

## 10  DCR summary

In [ ]:
import pandas as pd
def f(x): return float(sig(float(x)))
rows = [
 ("Bolt shear (dbl shear, F2280 N)",        f(DCR_bolt_shear),   "OK"),
 ("Bolt bearing / tearout (t=0.349)",       f(DCR_bolt_bearing), "OK"),
 ("Beam tension-flange rupture (F13)",      f(DCR_beam_flange),  "OK"),
 ("Fillet weld group - base elastic",       f(DCR_weld_base),    "OK"),
 ("Fillet weld group - K5 eff width",       f(DCR_weld_K5),      "GOVERNS (watch)"),
 ("Cover plate flexure",                    f(DCR_plate),        "OK"),
 ("Column side-wall yield (beta=1, approx)",f(DCR_col_wall),     "OK"),
 ("Stub C-section flexure",                 f(DCR_stub_flex),    "OK"),
 ("Stub C-section shear",                   f(DCR_stub_shear),   "OK"),
 ("Seat flange - 2-way + sharing",          f(DCR_seat_2way),    "OK"),
]
df = pd.DataFrame(rows, columns=["Limit state","DCR","Status"])
print(f"Stub open-section St-Venant torsion: tau = {sig(float(tau_t))} ksi "
      f"(add warping + flexural shear)")
df


## 11  Notes & export

- **Governing:** fillet weld under the K5 effective-width treatment (~0.65; approaches
  1.0 on the conservative flange-force interpretation) — consider 7/16" leg for margin.
- **Path A premise** underlies the bolt and seat checks; keep the couple in the bolts.
- **Verify separately:** stub open-section torsion combined with flexural shear;
  a clean Chapter K column check; beam `Sx` against the AISC table; the beam member
  under the combined H1 interaction.
- **Bolt shear** uses `Fnv = 68 ksi` (threads included, N); if threads are excluded
  (X, 84 ksi) all bolt DCRs drop ~19%.

Export via the uploaded helper (writes to the project `outputs/` folder):

In [ ]:
# export_notebook("html")   # or "pdf" (WeasyPrint backend on Colab)
export_notebook("html")